# AI PCB Inspector — Option B public-data pilot

Run this notebook in hosted Google Colab. It downloads datasets into Colab, audits labels, crops annotated solder-joint ROIs, trains MobileNetV3-Small, and saves artifacts to Drive. Nothing is downloaded to or trained on the project PC. PyTorch is already provided by Colab; this notebook does not install PyTorch/torchvision on your PC. The script automatically uses a Colab GPU if available and falls back to CPU otherwise. GPU is optional, though CPU training will be slower.

Before running: create Colab Secrets `KAGGLE_USERNAME`, `KAGGLE_KEY`, and `ROBOFLOW_API_KEY`; upload the companion `option_b_colab.py` when prompted. Never paste credentials into cells or commit them. Review category definitions and representative images before filling class maps. In particular, do not assume `exc_solder` means bridge or `STICKSOLDER` means solder bridge.

Public image-level splits do not prove physical-board independence. This is a feasibility pilot, not final project accuracy. Option B's actual OpenCV ROI locator and Pi runtime are separate integration work.

In [ ]:
%pip -q install kaggle roboflow scikit-learn pillow
import os, sys, subprocess
from pathlib import Path
from google.colab import userdata, files, drive

drive.mount('/content/drive')
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
uploaded = files.upload()
if 'option_b_colab.py' not in uploaded:
    raise FileNotFoundError('Upload the companion option_b_colab.py file.')
print('Secrets loaded from Colab Secrets; companion script uploaded.')

In [ ]:
subprocess.run([sys.executable, 'option_b_colab.py', 'download', '--root', '/content/pcb_inspector', '--roboflow-version', '1'], check=True)
subprocess.run([sys.executable, 'option_b_colab.py', 'inspect', '--root', '/content/pcb_inspector'], check=True)
from PIL import Image
from IPython.display import display
for preview in sorted(Path('/content/pcb_inspector/audit_previews').glob('*.jpg')):
    print(preview.name)
    display(Image.open(preview))

Inspect the preview images and category/count report from the previous cell. Verify the Roboflow terms and label definitions in your logged-in account. Then edit the maps below and set the Roboflow `license_confirmed` value to `True` in this cell only after confirming its terms. The defaults intentionally map nothing; training stops unless all three classes are supported by reviewed labels.

In [ ]:
import json
from pathlib import Path

# Replace category names only after checking their official definitions and sample images.
target_classes = ['good', 'solder_bridge', 'insufficient_solder']
class_maps = {
    'soldef': {},
    'roboflow': {},
}
# Example map shape only. Add exact category names only after checking definitions and images.
audit = json.loads(Path('/content/pcb_inspector/audit_summary.json').read_text())
roots = {'soldef': '/content/pcb_inspector/soldef', 'roboflow': '/content/pcb_inspector/roboflow'}
datasets = []
for source, entries in audit.items():
    if entries and class_maps[source]:
        dataset = {'name': source, 'root': roots[source], 'annotations': [entry['annotation'] for entry in entries], 'class_map': class_maps[source]}
        if source == 'soldef':
            dataset.update({'source_version': 'Kaggle v1', 'license': 'GPL-3.0 (Kaggle API metadata)', 'license_confirmed': True, 'source_url': 'https://www.kaggle.com/datasets/mauriziocalabrese/soldef-ai-pcb-dataset-for-defect-detection'})
        else:
            # Change license_confirmed to True only after verifying the Roboflow licence in your account.
            dataset.update({'source_version': 'Roboflow v1', 'license': 'Verify in account before training', 'license_confirmed': False, 'source_url': 'https://universe.roboflow.com/work-6qkmv/pcb-solder-joint'})
        datasets.append(dataset)
config = {'target_classes': target_classes, 'datasets': datasets}
Path('/content/pcb_inspector/training_config.json').write_text(json.dumps(config, indent=2))
print('Configured sources:', [(d['name'], d['annotations'], d['class_map']) for d in datasets])
print('Edit class_maps above using audited categories, then rerun this cell.')

Once the mapping is filled and saved, run the training cell. It groups crops by source image to prevent ROI leakage across splits, but public data do not identify physical boards.

In [ ]:
subprocess.run([sys.executable, 'option_b_colab.py', 'train', '--config', '/content/pcb_inspector/training_config.json', '--output', '/content/option_b_run', '--drive-output', '/content/drive/MyDrive/AI-PCB-Inspector/option_b_public_pilot', '--device', 'auto'], check=True)
# To force CPU even if Colab has a GPU, change '--device', 'auto' to '--device', 'cpu'.